In [1]:
import os

os.chdir('..')
os.getcwd()

'c:\\Users\\chouybat\\repositories\\vfm-quality-control-benchmark'

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact_manual
from PIL import Image

from vfm_quality_control_benchmark.data_types.enums import ModelName
from vfm_quality_control_benchmark.models.factory import Factory
from vfm_quality_control_benchmark.utils.tasks import to_anomaly_map

CONFIG = Path('src/vfm_quality_control_benchmark/configs/models.yaml')
models = {n: Factory.build_model(Factory.load_config(n, CONFIG)) for n in ModelName}

In [ ]:
# image: path relative to repo root; prompt: comma-separated
@interact_manual(
    model=list(ModelName),
    image='data/3CAD/3CAD/Aluminum_Camera_Cover/test/scratche/000001.png',
    prompt='scratch',
)
def show(model: ModelName, image: str, prompt: str) -> None:
    path = Path(image)
    img = Image.open(path).convert('RGB')
    pred = models[model].predict(img, [p.strip() for p in prompt.split(',')])
    amap, score = to_anomaly_map(pred)
    gt = path.parents[2] / 'ground_truth' / path.parent.name / path.name

    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    for a in ax:
        a.imshow(img)
        a.axis('off')
        if gt.exists():
            a.contour(np.array(Image.open(gt)) > 0, colors='lime')
    ax[1].imshow(amap, cmap='jet', alpha=0.5, vmin=0, vmax=1)
    ax[1].set_title(f'{model}: image score {score:.2f}')
    plt.show()